In [1]:
import hopsworks
import pandas as pd
import os
from dotenv import load_dotenv

load_dotenv()

# ── Connect ────────────────────────────────────────────────────────────────────
project = hopsworks.login(
    host=os.getenv("HOPSWORKS_HOST", "eu-west.cloud.hopsworks.ai"),
    api_key_value=os.getenv("HOPSWORKS_API_KEY")
)
fs      = project.get_feature_store()
fg      = fs.get_feature_group(name="aqi_features", version=1)
df      = fg.read(online=False)
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
df      = df.sort_values("timestamp").reset_index(drop=True)

print(f"Total rows:  {len(df)}")
print(f"Date range:  {df['timestamp'].min()} → {df['timestamp'].max()}")
print(f"Columns:     {len(df.columns)}")

# ── Basic stats ────────────────────────────────────────────────────────────────
print("\n--- AQI stats ---")
print(df["aqi"].describe().round(2))

# ── Check forecast column coverage ────────────────────────────────────────────
fc_cols = [c for c in df.columns if c.startswith("fc_")]
print(f"\n--- Forecast columns ({len(fc_cols)}) ---")
print(df[fc_cols].notna().sum().to_string())

# ── Rows WITH forecast features ───────────────────────────────────────────────
rows_with_fc = df[df["fc_pm25_24h"].notna()]
print(f"\n--- Rows with forecast features: {len(rows_with_fc)} ---")
print(rows_with_fc[["timestamp", "aqi", "fc_pm25_24h", "fc_dust_48h", "fc_uvi_72h"]].tail(10).to_string())

# ── Rows with real AQI (training eligible) ────────────────────────────────────
real_aqi = df[df["aqi"].notna()]
print(f"\n--- Rows with real AQI (training eligible): {len(real_aqi)} ---")
print(real_aqi[["timestamp", "aqi", "pm25", "pm10", "rolling_mean_24h"]].tail(10).to_string())

# ── Check NaN counts per column ───────────────────────────────────────────────
print("\n--- NaN counts per column ---")
null_counts = df.isnull().sum()
print(null_counts[null_counts > 0].to_string())

# ── AQI distribution by year ───────────────────────────────────────────────────
print("\n--- AQI mean by year ---")
print(df.groupby(df["timestamp"].dt.year)["aqi"].agg(["mean", "count", "min", "max"]).round(2))

# ── Latest 5 rows ─────────────────────────────────────────────────────────────
print("\n--- Latest 5 rows ---")
print(df.tail(5)[["timestamp", "aqi", "pm25", "pm10", "aqi_lag_1h", 
                   "rolling_mean_24h", "fc_pm25_24h", "fc_dust_48h"]].to_string())

# ── Earliest 5 rows ───────────────────────────────────────────────────────────
print("\n--- Earliest 5 rows ---")
print(df.head(5)[["timestamp", "aqi", "pm25", "pm10", "aqi_lag_1h",
                   "rolling_mean_24h", "fc_pm25_24h"]].to_string())

d:\Projects\Intern Project\AQI_Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-05-15 12:40:46,785 INFO: Initializing external client
2026-05-15 12:40:46,785 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-05-15 12:40:49,548 INFO: Python Engine initialized.

Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/31957
Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (1.94s) 
Total rows:  2340
Date range:  2026-01-29 00:00:00+00:00 → 2026-05-15 06:00:00+00:00
Columns:     48

--- AQI stats ---
count    2340.00
mean       82.78
std        24.08
min        14.58
25%        66.56
50%        78.34
75%        93.90
max       163.60
Name: aqi, dtype: float64

--- Forecast columns (21) ---
fc_pm25_24h    2339
fc_co_24h      2339
fc_no2_24h     2339
fc_so2_24h     2339
fc_o3_24h      2339
fc_dust_24h    2339
fc_uvi_24h     2339
fc_pm25_48h    2339
fc_co_48h      2339
fc_no2_48h     2339
fc_so2_48h     2339
fc_o3_48h      2339
fc_dust_48h    2339
fc_uvi_48h     2339
fc_pm25_72h    2339
fc_co_72h      2339


In [1]:
import hopsworks, os
from dotenv import load_dotenv
load_dotenv()

project = hopsworks.login(
    host=os.getenv("HOPSWORKS_HOST"),
    api_key_value=os.getenv("HOPSWORKS_API_KEY")
)
fs = project.get_feature_store()
fg = fs.get_feature_group("aqi_features", version=1)
df = fg.read(online=False)
print("Total rows:", len(df))
print("Latest timestamp:", df["timestamp"].max())
print("Oldest timestamp:", df["timestamp"].min())

d:\Projects\Intern Project\AQI_Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-05-08 16:32:20,833 INFO: Initializing external client
2026-05-08 16:32:20,833 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-05-08 16:32:23,540 INFO: Python Engine initialized.

Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/31957
Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (21.76s) 
Total rows: 2259
Latest timestamp: 2026-05-03 14:00:00+00:00
Oldest timestamp: 2026-01-29 00:00:00+00:00


In [2]:

import hopsworks, os
from dotenv import load_dotenv
load_dotenv()
project = hopsworks.login(host=os.getenv('HOPSWORKS_HOST'), api_key_value=os.getenv('HOPSWORKS_API_KEY'))
fs = project.get_feature_store()
fg = fs.get_feature_group('aqi_features', version=1)
import pandas as pd
df = fg.read(online=False)
print('Total rows:', len(df))
print('Latest timestamp:', df['timestamp'].max())

2026-05-08 17:20:55,206 INFO: Closing external client and cleaning up certificates.
2026-05-08 17:20:55,214 INFO: Connection closed.
2026-05-08 17:20:55,216 INFO: Initializing external client
2026-05-08 17:20:55,217 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-05-08 17:20:57,739 INFO: Python Engine initialized.

Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/31957
Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (1.82s) 
Total rows: 2261
Latest timestamp: 2026-05-08 10:00:00+00:00


In [3]:
from dotenv import load_dotenv
import os
import pandas as pd
import hopsworks
from aqi_feature_utils import pm25_to_aqi

load_dotenv()
project = hopsworks.login(
    host=os.getenv("HOPSWORKS_HOST", "eu-west.cloud.hopsworks.ai"),
    api_key_value=os.getenv("HOPSWORKS_API_KEY")
)
fs = project.get_feature_store()
fg = fs.get_feature_group('aqi_features', version=1)
df = fg.read(online=False)
df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)

# ── Find MAX values ────────────────────────────────────────────────────────────
print(f"MAX PM2.5: {df['pm25'].max():.1f}")
print(f"MAX stored AQI: {df['aqi'].max():.1f}")

# ── Find rows with highest PM2.5 ────────────────────────────────────────────────
top_pm25 = df.nlargest(5, 'pm25')[['timestamp', 'pm25', 'aqi']]
top_pm25['aqi_calc'] = top_pm25['pm25'].apply(pm25_to_aqi)
print("\n--- Top 5 PM2.5 rows ---")
print(top_pm25.to_string())

# ── Find rows with highest AQI ────────────────────────────────────────────────
top_aqi = df.nlargest(5, 'aqi')[['timestamp', 'pm25', 'aqi']]
top_aqi['aqi_calc'] = top_aqi['pm25'].apply(pm25_to_aqi)
print("\n--- Top 5 AQI rows ---")
print(top_aqi.to_string())


2026-05-15 12:45:49,028 INFO: Closing external client and cleaning up certificates.
2026-05-15 12:45:49,031 INFO: Connection closed.
2026-05-15 12:45:49,033 INFO: Initializing external client
2026-05-15 12:45:49,034 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-05-15 12:45:51,534 INFO: Python Engine initialized.

Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/31957
Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (8.24s) 
MAX PM2.5: 79.9
MAX stored AQI: 163.6

--- Top 5 PM2.5 rows ---
                     timestamp  pm25         aqi    aqi_calc
1717 2026-02-03 17:00:00+00:00  79.9  163.598525  163.598525
1229 2026-02-05 17:00:00+00:00  79.6  163.443625  163.443625
742  2026-02-05 16:00:00+00:00  79.2  163.237092  163.237092
957  2026-02-03 16:00:00+00:00  75.9  161.533193  161.533193
674  2026-02-03 18:00:00+00:00  75.2  161.171760  161.171760

--- Top 5 AQI rows ---
                     timestamp  pm25         a

## DATA DUPLICATION CHECKING 

In [1]:
import hopsworks
import pandas as pd
import os
from dotenv import load_dotenv

load_dotenv()

# ── Connect ───────────────────────────────────────────────────────────────────
project = hopsworks.login(
    host=os.getenv("HOPSWORKS_HOST", "eu-west.cloud.hopsworks.ai"),
    api_key_value=os.getenv("HOPSWORKS_API_KEY")
)
fs = project.get_feature_store()
fg = fs.get_feature_group("aqi_features", version=1)

# ── Read offline store ─────────────────────────────────────────────────────────
df = fg.read(online=False)
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
df = df.sort_values("timestamp").reset_index(drop=True)

# ── Basic info ─────────────────────────────────────────────────────────────────
print("Total rows     :", len(df))
print("Oldest         :", df["timestamp"].min())
print("Latest         :", df["timestamp"].max())
print("Columns        :", len(df.columns))

# ── Latest 10 rows ────────────────────────────────────────────────────────────
print("\n=== LATEST 10 ROWS ===")
cols = ["timestamp", "aqi", "pm25", "pm10", "temperature", "humidity", "wind_speed"]
df[cols].tail(10)

d:\Projects\Intern Project\AQI_Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-05-15 16:22:47,174 INFO: Initializing external client
2026-05-15 16:22:47,174 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-05-15 16:22:49,901 INFO: Python Engine initialized.

Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/31957
Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (2.10s) 
Total rows     : 2341
Oldest         : 2026-01-29 00:00:00+00:00
Latest         : 2026-05-15 09:00:00+00:00
Columns        : 48

=== LATEST 10 ROWS ===


,timestamp,aqi,pm25,pm10,temperature,humidity,wind_speed
2331,2026-05-14 11:00:00+00:00,53.0,12.9,22.9,31.6,66.0,18.5
2332,2026-05-14 14:00:00+00:00,54.0,13.3,24.3,29.0,80.0,16.9
2333,2026-05-14 16:00:00+00:00,41.0,9.8,17.3,28.6,81.0,16.4
2334,2026-05-14 18:00:00+00:00,41.0,9.9,18.0,28.4,83.0,14.9
2335,2026-05-14 20:00:00+00:00,38.0,9.1,18.0,28.2,82.0,13.2
2336,2026-05-14 22:00:00+00:00,36.0,8.7,19.5,28.0,83.0,14.6
2337,2026-05-14 23:00:00+00:00,38.0,9.2,22.6,28.1,83.0,15.3
2338,2026-05-15 02:00:00+00:00,49.0,11.7,28.6,28.8,81.0,15.9
2339,2026-05-15 06:00:00+00:00,56.0,14.4,38.3,32.3,60.0,17.9
2340,2026-05-15 09:00:00+00:00,55.0,14.2,34.3,32.9,59.0,17.8


In [2]:
# ── Check for duplicate timestamps ────────────────────────────────────────────
dupes = df[df.duplicated(subset=["timestamp"], keep=False)]
print("Duplicate timestamps:", len(dupes))
if not dupes.empty:
    print(dupes[["timestamp", "aqi"]].head(10))

Duplicate timestamps: 0


In [3]:
# ── Check for gaps > 2 hours ──────────────────────────────────────────────────
df["gap"] = df["timestamp"].diff()
large_gaps = df[df["gap"] > pd.Timedelta(hours=2)][["timestamp", "gap"]]
print(f"Gaps > 2 hours: {len(large_gaps)}")
large_gaps.tail(10)

Gaps > 2 hours: 32


,timestamp,gap
2322,2026-05-13 12:00:00+00:00,0 days 03:00:00
2323,2026-05-13 15:00:00+00:00,0 days 03:00:00
2324,2026-05-13 18:00:00+00:00,0 days 03:00:00
2328,2026-05-14 02:00:00+00:00,0 days 03:00:00
2329,2026-05-14 06:00:00+00:00,0 days 04:00:00
2330,2026-05-14 09:00:00+00:00,0 days 03:00:00
2332,2026-05-14 14:00:00+00:00,0 days 03:00:00
2338,2026-05-15 02:00:00+00:00,0 days 03:00:00
2339,2026-05-15 06:00:00+00:00,0 days 04:00:00
2340,2026-05-15 09:00:00+00:00,0 days 03:00:00


In [4]:
# ── Check if recent rows have same AQI (stuck/stale data) ─────────────────────
recent = df.tail(24).copy()
unique_aqi = recent["aqi"].nunique()
print(f"Unique AQI values in last 24 rows: {unique_aqi}")
print(f"AQI std dev in last 24 rows      : {recent['aqi'].std():.2f}")

if unique_aqi <= 3:
    print("⚠️  WARNING: Very few unique AQI values — data may be stuck/repeating")
else:
    print("✅  AQI values are varying normally")

recent[["timestamp", "aqi", "pm25", "temperature"]].tail(24)

Unique AQI values in last 24 rows: 17
AQI std dev in last 24 rows      : 18.66
✅  AQI values are varying normally


,timestamp,aqi,pm25,temperature
2317,2026-05-12 22:00:00+00:00,73.0,22.5,28.1
2318,2026-05-13 00:00:00+00:00,73.0,22.6,27.3
2319,2026-05-13 04:00:00+00:00,80.0,26.0,32.5
2320,2026-05-13 06:00:00+00:00,106.0,37.6,35.0
2321,2026-05-13 09:00:00+00:00,93.0,32.0,36.5
2322,2026-05-13 12:00:00+00:00,48.0,11.6,34.4
2323,2026-05-13 15:00:00+00:00,70.0,21.3,31.8
2324,2026-05-13 18:00:00+00:00,74.0,23.2,30.6
2325,2026-05-13 20:00:00+00:00,74.0,22.8,29.5
2326,2026-05-13 22:00:00+00:00,54.0,13.4,28.6


In [5]:
# ── Hourly row count per day (should be ~24 per day) ─────────────────────────
daily_counts = df.groupby(df["timestamp"].dt.date).size().reset_index()
daily_counts.columns = ["date", "row_count"]
print("=== ROWS PER DAY (last 10 days) ===")
print(daily_counts.tail(10).to_string(index=False))

=== ROWS PER DAY (last 10 days) ===
      date  row_count
2026-05-02         18
2026-05-03          9
2026-05-08          9
2026-05-09         17
2026-05-10         15
2026-05-11          9
2026-05-12          9
2026-05-13         10
2026-05-14         10
2026-05-15          3


In [6]:
import pandas as pd
from datetime import timezone

now_utc = pd.Timestamp.now(tz="UTC")
latest = df["timestamp"].max()
gap_hours = (now_utc - latest).total_seconds() / 3600
print(f"Current UTC time : {now_utc.strftime('%Y-%m-%d %H:%M')}")
print(f"Latest data point: {latest.strftime('%Y-%m-%d %H:%M')}")
print(f"Data freshness   : {gap_hours:.1f} hours old")

if gap_hours > 3:
    print("⚠️  Data is stale — pipeline may not have run recently")
else:
    print("✅  Data is fresh")

Current UTC time : 2026-05-15 11:31
Latest data point: 2026-05-15 09:00
Data freshness   : 2.5 hours old
✅  Data is fresh
